# CAASPP School-Level DFS — companion notebook

Calculates **Smarter Balanced-only DFS** (not official California Dashboard DFS). Upload districtwide **All Student Groups** files: 2021–23 caret-delimited, or 2024–26 caret/fixed-width TXT/ZIP files. Legacy 2022–23 fixed-width is also supported. All schools, including Mariposa, remain included. Older files may lack school names, so supply a newer file to backfill them automatically; otherwise codes are shown. 

Grade-level DFS = mean scale score minus grade/subject Level 3 cutoff; school DFS = student-with-scores weighted average. Check `Complete_3_5` before comparing three-grade school averages.

References: [CAASPP 2026 format](https://caaspp-elpac.ets.org/caaspp/ResearchFileFormatSB?lstTestYear=2026), [CAASPP 2023 format](https://caaspp-elpac.ets.org/caaspp/ResearchFileFormatSB?lstTestYear=2023).


In [ ]:
#@title Step 1: Upload CAASPP TXT/CSV/ZIP files
from google.colab import files
uploaded=files.upload()
if not uploaded: raise ValueError('Please upload at least one research file')


In [ ]:
#@title Step 2: Calculate school-level DFS for legacy and newer files
import csv
from zipfile import ZipFile
from io import BytesIO, StringIO
import pandas as pd

CUTS = {'01': {'03':2432, '04':2473, '05':2502},
        '02': {'03':2436, '04':2485, '05':2528}}
SUBJECT = {'01':'ELA', '02':'Math'}
COUNTY_CODE = '19' #@param {type:'string'}
DISTRICT_CODE = '64683' #@param {type:'string'}

def f(line, a, b):
    return line[a-1:b].strip()

def inputs(uploaded):
    for name, blob in uploaded.items():
        if name.lower().endswith('.zip'):
            with ZipFile(BytesIO(blob)) as z:
                for member in z.namelist():
                    if member.lower().endswith(('.txt','.csv')) and not member.startswith('__MACOSX'):
                        yield member, z.read(member).decode('utf-8-sig', errors='replace')
        elif name.lower().endswith(('.txt','.csv')):
            yield name, blob.decode('utf-8-sig', errors='replace')
        else:
            raise ValueError(f'Unsupported file: {name}')

def get(row, *names):
    lower = {str(k).strip().lower().replace('_',' '): v for k,v in row.items()}
    for name in names:
        val = lower.get(name.lower())
        if val is not None:
            return str(val).strip()
    return ''

def parse_rows(source, content):
    lines = [s for s in content.splitlines() if s.strip()]
    if not lines:
        return
    first = lines[0]
    if '^' in first or (',' in first and not (len(first)>124 and f(first,121,124).isdigit())):
        sep = '^' if '^' in first else ','
        reader = csv.DictReader(StringIO(content), delimiter=sep)
        if not reader.fieldnames:
            raise ValueError(f'{source}: no header in delimited file')
        for row in reader:
            yield {
                'county':get(row,'County Code'),
                'district':get(row,'District Code'),
                'school_code':get(row,'School Code'),
                'school':get(row,'School Name'),
                'year':get(row,'Test Year'),
                'type':get(row,'Type ID'),
                'group':get(row,'Student Group ID','Subgroup ID'),
                'grade':get(row,'Grade'),
                'test':get(row,'Test ID'),
                'count':get(row,'Students with Scores','Total Students Tested with Scores',
                            'Total Tested with Scores'),
                'mean':get(row,'Mean Scale Score')
            }
    else:
        named = len(first)>=201 and f(first,121,124).isdigit()
        legacy = len(first)>=101 and f(first,19,22).isdigit()
        if not named and not legacy:
            raise ValueError(f'{source}: unknown fixed-width layout')
        for line in lines:
            if named:
                yield {
                    'county':f(line,1,2),'district':f(line,3,7),
                    'school_code':f(line,48,54),'school':f(line,55,114),
                    'type':f(line,115,116),'year':f(line,121,124),
                    'test':f(line,126,127),'group':f(line,128,130),
                    'grade':f(line,131,132),'count':f(line,151,159),
                    'mean':f(line,160,165)
                }
            else:
                yield {
                    'county':f(line,1,2),'district':f(line,3,7),
                    'school_code':f(line,8,14),'school':'',
                    'type':f(line,174,175) if len(line)>=175 else '',
                    'year':f(line,19,22),'group':f(line,23,25),
                    'grade':f(line,41,42),'test':f(line,43,44),
                    'count':f(line,95,101),'mean':f(line,59,64)
                }

rows = []
for source,content in inputs(uploaded):
    for r in parse_rows(source,content):
        if r['county'].zfill(2)!=COUNTY_CODE.zfill(2) or r['district'].zfill(5)!=DISTRICT_CODE.zfill(5):
            continue
        if r['group'].zfill(3)!='001':
            continue
        # Legacy files can omit Type ID; those rows are filtered by their school code.
        if r['type'] and r['type'].zfill(2) not in ('07','09','10'):
            continue
        if r['school_code'].zfill(7)=='0000000':
            continue
        grade,test=r['grade'].zfill(2),r['test'].zfill(2)
        if grade not in ('03','04','05') or test not in CUTS:
            continue
        try:
            count=int(r['count'].replace(',',''))
            mean=float(r['mean'].replace(',',''))
            year=int(r['year'])
        except (ValueError,TypeError):
            continue  # suppressed/missing score records
        if count<=0 or not (2015<=year<=2030):
            continue
        rows.append({
            'Test Year':year,'School Code':r['school_code'].zfill(7),
            'School':r['school'],'Grade':int(grade),'Subject':SUBJECT[test],
            'Students With Scores':count,'Mean Scale Score':mean,
            'Level 3 Cutoff':CUTS[test][grade],
            'DFS':mean-CUTS[test][grade],'Source File':source
        })
detail=pd.DataFrame(rows)
if detail.empty:
    raise ValueError('No eligible school-level grade 3–5 records found. Check district codes and file type.')

# Older districtwide files may not embed school names. Fill from other years when possible.
name_lookup=(detail.loc[detail['School'].ne(''),['School Code','School']]
             .drop_duplicates('School Code').set_index('School Code')['School'].to_dict())
detail['School']=detail.apply(
    lambda r: r['School'] or name_lookup.get(r['School Code'], 'School '+r['School Code']),axis=1)

keys=['Test Year','School Code','Grade','Subject']
if detail.duplicated(keys).any():
    duplicates=detail.loc[detail.duplicated(keys,keep=False),keys+['Source File']]
    raise ValueError('Duplicate year/school/grade/subject rows:\n'+duplicates.to_string(index=False))

detail['Weighted DFS']=detail['DFS']*detail['Students With Scores']
school=(detail.groupby(['Test Year','School Code','School','Subject'],as_index=False)
        .agg(Students_With_Scores=('Students With Scores','sum'),
             Weighted_Total=('Weighted DFS','sum'),
             Grades_Included=('Grade','nunique')))
school['DFS']=(school['Weighted_Total']/school['Students_With_Scores']).round(1)
school['Complete_3_5']=school['Grades_Included'].eq(3)
school=school.drop(columns=['Weighted_Total']).sort_values(['Test Year','Subject','School'])
detail.drop(columns=['Weighted DFS']).to_csv('caaspp_dfs_grade_detail.csv',index=False)
school.to_csv('caaspp_dfs_school_summary.csv',index=False)
print('Calculated Smarter Balanced DFS only (not official Dashboard DFS). No school exclusions.')
print('Test years:',sorted(school['Test Year'].unique()))
display(school[['Test Year','School','Subject','Students_With_Scores','Grades_Included','Complete_3_5','DFS']])


In [ ]:
#@title Step 3: Download tables
from google.colab import files
files.download('caaspp_dfs_grade_detail.csv')
files.download('caaspp_dfs_school_summary.csv')


## Interpretation

Create trend charts from `caaspp_dfs_school_summary.csv` using Test Year on the x-axis and DFS on the y-axis. School names in old years come from any newer file uploaded in the same session. Missing grade coverage may distort comparisons. This notebook does not incorporate California Alternate Assessment results or Dashboard participation rules, and doesn't change the original percent-met notebook.
